# A2 - Knowledge-Base Demo
Show OCR quality on a sample and one working retrieval example.

## Building a demo knowledge base from the human-gold subset

The full corpus is 546 distinct pages (~65,500 words, est.). Stage 3's only measured reader
(Qwen2.5-VL-3B) is **not viable at that scale**: mean `ocr_f1` **0.022** on held-out gold
pages -- it confabulates fluent, unrelated Bangla prose rather than reading the handwriting
(`reports/ocr_baseline.json`, see the OCR-quality cell below). Indexing that output at
full-corpus scale would fill the store with invented text, which is worse than an empty
store.

So **this demo index is built from the pages that already have a real, human-typed gold
transcription** in `grading_kit/labels.jsonl` (`status: "done"`), not from the automated
reader's output. That's the honest thing to index right now: it proves `chunk -> embed ->
index -> retrieve` works end-to-end on real, human-verified Bangla legal text, while being
explicit that it covers a small fraction of the corpus, not the whole thing.

In [1]:
from __future__ import annotations

import csv
import json
import os
import sys
from pathlib import Path

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
# config.yaml paths (index.path, embed.cache_dir, ...) are relative and assume the process
# runs from the repo root, same as scripts/run_ingest.py. Jupyter/nbconvert start this
# kernel with cwd=notebooks/, so without this, index files would silently land in
# notebooks/data/... instead of the real data/... at the repo root.
os.chdir(ROOT)

# faiss and torch (pulled in by sentence-transformers, used by embed.py) each bundle their
# own OpenMP runtime; loading both in one process on Windows aborts with "OMP: Error #15"
# unless this is set before faiss is imported. Set here, not in store.py, to keep this a
# notebook-only change.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from doc_agent import config
from doc_agent.contracts import Chunk
from doc_agent.index import chunk as chunk_mod
from doc_agent.index import embed, store

cfg = config.load(ROOT / "configs" / "config.yaml")

# page_id -> human-typed gold text, for labels a human has actually finished (same rule as
# scripts/score_heldout.py:load_gold -- a "TODO" row must not silently count as content).
labels_path = ROOT / "grading_kit" / "labels.jsonl"
gold: dict[str, str] = {}
with labels_path.open(encoding="utf-8") as fh:
    for line in fh:
        if not line.strip():
            continue
        record = json.loads(line)
        text = (record.get("text") or "").strip()
        if record.get("status") == "done" and text:
            gold[record["page_id"]] = text

# page_id -> doc_id, straight from the real corpus grouping. .get() with a page_id fallback,
# not indexing: a gold page can legitimately be absent (e.g. a held-out page the main-corpus
# duplicate sweep later removed) without that being an error.
groups: dict[str, str] = {}
with (ROOT / "data" / "deed_groups.csv").open(encoding="utf-8", newline="") as fh:
    for row in csv.DictReader(fh):
        groups[row["page_id"]] = row["doc_id"]

source_chunks = [
    Chunk(
        id=f"{page_id}#gold",
        doc_id=groups.get(page_id, page_id),
        text=text,
        page_ids=[page_id],
    )
    for page_id, text in sorted(gold.items())
]

split_chunks = chunk_mod.split(source_chunks, cfg)
vectors = embed.encode(split_chunks, cfg)
store.build(split_chunks, vectors, cfg)

TOTAL_CORPUS_PAGES = 546  # data/provenance.md
TOTAL_CORPUS_WORDS = 65500  # data/provenance.md, estimate
subset_words = sum(len(t.split()) for t in gold.values())

print(f"demo index built from {len(gold)} human-gold pages -- NOT the full automated corpus")
print(f"  chunks indexed:      {len(split_chunks)}")
print(f"  embedding dimension: {vectors.shape[1]}")
print(f"  index type:          {cfg['index']['type']}")
print(
    f"  corpus coverage:     {len(gold)}/{TOTAL_CORPUS_PAGES} pages "
    f"({len(gold) / TOTAL_CORPUS_PAGES:.1%}), {subset_words}/{TOTAL_CORPUS_WORDS} words "
    f"({subset_words / TOTAL_CORPUS_WORDS:.1%})"
)
print(f"  pages used: {sorted(gold.keys())}")

C:\Users\Asus\AppData\Roaming\Python\Python310\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

C:\Users\Asus\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\cse\4-1\429\doc-agent-12\.hf_cache\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

demo index built from 17 human-gold pages -- NOT the full automated corpus
  chunks indexed:      119
  embedding dimension: 384
  index type:          faiss:flat
  corpus coverage:     17/546 pages (3.1%), 2398/65500 words (3.7%)
  pages used: ['dolil_100', 'dolil_101', 'dolil_13', 'dolil_20', 'dolil_208', 'dolil_212', 'dolil_348', 'dolil_38', 'dolil_49', 'dolil_60', 'dolil_605', 'dolil_65', 'dolil_71', 'dolil_82', 'dolil_93', 'dolil_98', 'dolil_99']


## OCR quality on a sample

The real, measured number behind the decision above -- not the demo index itself, which
deliberately bypasses OCR by indexing human gold text instead (see why above).

In [2]:
baseline_path = ROOT / "reports" / "ocr_baseline.json"
baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
summary = baseline["summary"]

print(f"model: {baseline['model']}")
print(
    f"pages: {summary['pages']}  mean F1: {summary['mean_f1']:.3f}  "
    f"median F1: {summary['median_f1']:.3f}  "
    f"range: {summary['min_f1']:.3f}-{summary['max_f1']:.3f}  "
    f"truncated: {summary['truncated_pages']}/{summary['pages']}"
)
print()
print("Target is F1 >= 0.92. The reader is not misreading the handwriting -- it is not")
print("reading it: it confabulates fluent, unrelated Bangla prose about a document-shaped")
print("image and then repeats until the token cap. That gap is exactly why the retrieval")
print("demo below is built from human gold text instead of this reader's output.")

model: Qwen/Qwen2.5-VL-3B-Instruct
pages: 7  mean F1: 0.022  median F1: 0.000  range: 0.000-0.076  truncated: 7/7

Target is F1 >= 0.92. The reader is not misreading the handwriting -- it is not
reading it: it confabulates fluent, unrelated Bangla prose about a document-shaped
image and then repeats until the token cap. That gap is exactly why the retrieval
demo below is built from human gold text instead of this reader's output.


## Retrieval demo
Run a natural Bangla question against the seeded FAISS index, print the retrieved chunk details, and assert that the expected page is retrieved.

In [3]:
from __future__ import annotations

import os
import sys
from pathlib import Path

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

from doc_agent import config
from doc_agent.index import embed, store

query = "দলিলে টেনু সাব্বির পিতার নাম কী?"
expected_page_id = "dolil_605"
k = 3

cfg = config.load(ROOT / "configs" / "config.yaml")
loaded = store.load(cfg)
query_vector = embed.encode_query(query, cfg)
scores, indices = loaded["index"].search(query_vector, k)

top_idx = int(indices[0][0])
top_score = float(scores[0][0])
top_record = loaded["metadata"][top_idx]

print(f"query: {query}")
print(f"expected_page_id: {expected_page_id}")
print(f"retrieved_page_id: {top_record['page_id']}")
print(f"chunk_id: {top_record['chunk_id']}")
print(f"score: {top_score:.6f}")
print("chunk:")
print(top_record["chunk_text"])

assert top_record["page_id"] == expected_page_id


query: দলিলে টেনু সাব্বির পিতার নাম কী?
expected_page_id: dolil_605
retrieved_page_id: dolil_605
chunk_id: dolil_605#c0068
score: 0.879938
chunk:
মিয়া পিতা টেনু সাব্বির শা:

সুরবাড়ি থানা জয়দেবপুর জিলা ঢাকা

[?] গাজীপুর ধর্ম ইসলাম ব্যবসা

জোতদারী দলিল গ্রহীতাগণ ১। টেনু

সাব্বির পিতা মৃত পাহালী মাঝি

সাং: সুরাবাড়ী থানা জয়দেবপুর

গাজীপুর জিলা ঢাকা ধর্ম ইসলাম

ব্যবসা জোতদারী দলিল দাতা

স্থিতিবান বায়তি জোতদার [?]

এত্তয়াজ হেবা নামা পত্র দলিল

সিদ: কার্যজ্ঞানে যেহেতু নিম্ন তা

ছিল বনিত জমিতে আমি দলিল

পত্র সিদ: কার্যজ্ঞানে যেহেতু নি
